In [0]:
import json
from pyspark.sql.functions import * 
from pyspark.sql.types import * 


In [0]:
df = spark.read.format("delta").option("multiline", "true").load("/Volumes/ap/bi/cricket_api/broze_cricket/")

In [0]:
display(df)

In [0]:
raw_json = df.select("raw_json").collect()[0]['raw_json']
raw_json


In [0]:
api_data = json.loads(raw_json)
api_data

In [0]:
matches = api_data.get("data",[])
matches

In [0]:
print("total matches:", len(matches))

In [0]:
matches[0]

In [0]:
silver_rows=[]
for match in matches:
    teams = match.get('teams', [])
    team1 = teams[0] if len(teams) > 0 else None
    team2 = teams[1] if len(teams) > 1 else None
    silver_rows.append({
        'id': match.get('id'),
        'name': match.get('name'),
        'matchType': match.get('matchType'),
        'status': match.get('status'),
        'venue': match.get('venue'),
        'date': match.get('date'),
        'team1': team1,
        'team2': team2,
        'matchStarted': match.get('matchStarted'),
        'matchEnded': match.get('matchEnded')
    })
silver_rows



In [0]:
from pyspark.sql.functions import current_timestamp,col
silver_schema = StructType([
    StructField('id', StringType(), True),
    StructField('name', StringType(), True),
    StructField('matchType', StringType(), True),
    StructField('status', StringType(), True),
    StructField('venue', StringType(), True),
    StructField('date', StringType(), True),
    StructField('team1', StringType(), True),
    StructField('team2', StringType(), True),
    StructField('matchStarted', BooleanType(), True),
    StructField('matchEnded', BooleanType(), True)
])
silver_df = spark.createDataFrame(silver_rows, schema=silver_schema)\
    .withColumn("match_date",to_date(col("date")))\
    .withColumn("load_date",current_timestamp())
display(silver_df)

In [0]:
silver_df.write.mode("overwrite").format("delta").save("/Volumes/ap/bi/cricket_api/silver_cricket/")
print("silevr table created successfully")

In [0]:
silver_df.createOrReplaceTempView("silver_cricket_view")

display(spark.sql("SELECT * FROM silver_cricket_view"))